# 全モデル比較評価ノートブック

USDJPY 5分足方向予測 - PatchTST / iTransformer / CNN ベースライン 比較評価

**Kaggle commit mode 対応**: このノートブックは Save & Run All で実行完了します。

- テストデータ入力: `/kaggle/input/pearless-usdjpy-m5/` の `X_test.npy` / `y_test.npy`
- モデル入力:
  - `/kaggle/input/pearless-patchtst-model/best_model.pt` (PatchTST)
  - `/kaggle/input/pearless-itransformer-model/best_model.pt` (iTransformer)
  - `/kaggle/input/pearless-cnn-model/best_model.pt` (CNN)
- 出力: `/kaggle/working/evaluation_results_all_{timestamp}.csv`（AC-015/AC-021）
- wandb 不使用（PRD wandb スコープ外確定）

**PRD 成功基準**:
1. UP/DOWN F1 スコア: PatchTST または iTransformer が CNN を 5pt 以上上回る
2. 高信頼度的中率 (Precision@0.8): 70% 以上
3. 推論レイテンシ: 50ms 未満（AC-017）

In [ ]:
# 依存パッケージのインストール（Kaggle 環境向け）
# Kaggle にはデフォルトで PyTorch がインストール済みのため、
# 追加パッケージのみインストールする
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn"],
    check=True
)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

# Kaggle 環境では /kaggle/input/{dataset-slug}/ にデータが配置される
DATASET_DIR = Path("/kaggle/input/pearless-usdjpy-m5")
WORKING_DIR = Path("/kaggle/working")

# 各モデルのチェックポイントが格納された入力 Dataset ディレクトリ
PATCHTST_MODEL_DIR = Path("/kaggle/input/pearless-patchtst-model")
ITRANSFORMER_MODEL_DIR = Path("/kaggle/input/pearless-itransformer-model")
CNN_MODEL_DIR = Path("/kaggle/input/pearless-cnn-model")

# ローカル環境でのフォールバック（テスト実行時）
if not DATASET_DIR.exists():
    DATASET_DIR = Path("../data")
if not PATCHTST_MODEL_DIR.exists():
    PATCHTST_MODEL_DIR = Path("../data")
if not ITRANSFORMER_MODEL_DIR.exists():
    ITRANSFORMER_MODEL_DIR = Path("../data")
if not CNN_MODEL_DIR.exists():
    CNN_MODEL_DIR = Path("../data")

# ソースコードを sys.path に追加（Kaggle でコードをコピーした場合の対応）
REPO_ROOT = Path("/kaggle/working/pearless")
if not REPO_ROOT.exists():
    REPO_ROOT = Path("..")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Working dir: {WORKING_DIR}")
print(f"PatchTST model dir: {PATCHTST_MODEL_DIR}")
print(f"iTransformer model dir: {ITRANSFORMER_MODEL_DIR}")
print(f"CNN model dir: {CNN_MODEL_DIR}")

In [ ]:
# テストデータ読み込み（AC-015: X_test.npy / y_test.npy を使用）
X_test: np.ndarray = np.load(DATASET_DIR / "X_test.npy")
y_test: np.ndarray = np.load(DATASET_DIR / "y_test.npy")

print(f"X_test shape: {X_test.shape}, dtype: {X_test.dtype}")
print(f"y_test shape: {y_test.shape}, dtype: {y_test.dtype}")

# shape 検証（AC-002）
assert X_test.ndim == 3 and X_test.shape[1] == 60 and X_test.shape[2] == 16, \
    f"X_test shape 不正: {X_test.shape}"

# クラス分布確認
unique, counts = np.unique(y_test, return_counts=True)
print("\nクラス分布 (y_test):")
for cls, cnt in zip(unique, counts):
    label = ["UP", "DOWN", "NEUTRAL"][cls]
    print(f"  {label}({cls}): {cnt} ({cnt/len(y_test)*100:.1f}%)")

In [ ]:
# 各モデルのロードとインスタンス生成
from models.cnn import CNNModel
from models.itransformer import iTransformer
from models.patchtst import PatchTST


def load_patchtst(model_dir: Path, device: torch.device) -> PatchTST:
    """PatchTST モデルをロードする。"""
    model = PatchTST(
        seq_len=60,
        n_features=16,
        patch_len=6,
        stride=6,
        d_model=128,
        n_heads=8,
        n_layers=3,
        dim_ff=256,
        dropout=0.2,
        n_classes=3,
    )
    model_path = model_dir / "best_model.pt"
    assert model_path.exists(), f"PatchTST モデルが見つかりません: {model_path}"
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.to(device)
    return model


def load_itransformer(model_dir: Path, device: torch.device) -> iTransformer:
    """iTransformer モデルをロードする。"""
    model = iTransformer(
        seq_len=60,
        n_features=16,
        d_model=128,
        n_heads=8,
        n_layers=3,
        dim_ff=256,
        dropout=0.2,
        n_classes=3,
    )
    model_path = model_dir / "best_model.pt"
    assert model_path.exists(), f"iTransformer モデルが見つかりません: {model_path}"
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.to(device)
    return model


def load_cnn(model_dir: Path, device: torch.device) -> CNNModel:
    """CNN ベースラインモデルをロードする。"""
    model = CNNModel(
        seq_len=60,
        n_features=16,
        n_classes=3,
    )
    model_path = model_dir / "best_model.pt"
    assert model_path.exists(), f"CNN モデルが見つかりません: {model_path}"
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.to(device)
    return model


patchtst_model = load_patchtst(PATCHTST_MODEL_DIR, DEVICE)
itransformer_model = load_itransformer(ITRANSFORMER_MODEL_DIR, DEVICE)
cnn_model = load_cnn(CNN_MODEL_DIR, DEVICE)

print("全モデルのロード完了")
print(f"  PatchTST     パラメータ数: {sum(p.numel() for p in patchtst_model.parameters()):,}")
print(f"  iTransformer パラメータ数: {sum(p.numel() for p in itransformer_model.parameters()):,}")
print(f"  CNN          パラメータ数: {sum(p.numel() for p in cnn_model.parameters()):,}")

In [ ]:
# 全モデルの評価実行（AC-015: evaluate_model() を使用）
from evaluate import compare_models, evaluate_model

print("評価開始...")

patchtst_result = evaluate_model(patchtst_model, X_test, y_test, DEVICE)
print(f"PatchTST 評価完了: accuracy={patchtst_result['accuracy']:.4f}, f1_macro={patchtst_result['f1_macro']:.4f}")

itransformer_result = evaluate_model(itransformer_model, X_test, y_test, DEVICE)
print(f"iTransformer 評価完了: accuracy={itransformer_result['accuracy']:.4f}, f1_macro={itransformer_result['f1_macro']:.4f}")

cnn_result = evaluate_model(cnn_model, X_test, y_test, DEVICE)
print(f"CNN 評価完了: accuracy={cnn_result['accuracy']:.4f}, f1_macro={cnn_result['f1_macro']:.4f}")

In [ ]:
# compare_models() で比較 DataFrame を生成（AC-021: CNN 列が欠損なく存在）
results = {
    "patchtst": patchtst_result,
    "itransformer": itransformer_result,
    "cnn": cnn_result,
}

df_comparison = compare_models(results)

# AC-021: 全モデルの行が揃っていることを検証
assert "patchtst" in df_comparison["model"].values, "patchtst が比較表に存在しない"
assert "itransformer" in df_comparison["model"].values, "itransformer が比較表に存在しない"
assert "cnn" in df_comparison["model"].values, "cnn が比較表に存在しない"
print("AC-021 OK: 全モデル（patchtst / itransformer / cnn）が比較表に存在する")

print("\n=== モデル比較表 ===")
print(df_comparison.to_string(index=False))

In [ ]:
# 高信頼度的中率（Precision@0.8）の計算（AC-016/AC-015）
# evaluate_model() は基本メトリクスのみ返すため、threshold 付き評価を追加実行
from evaluate import compute_metrics

THRESHOLD = 0.8


def compute_threshold_metrics(
    model: object,
    X: np.ndarray,
    y: np.ndarray,
    device: torch.device,
    threshold: float,
) -> dict:
    """モデルを eval モードで推論し、threshold 付きメトリクスを返す。"""
    model.eval()
    x_tensor = torch.from_numpy(X.astype(np.float32)).to(device)
    with torch.no_grad():
        y_prob_tensor = model(x_tensor)
    y_prob = y_prob_tensor.cpu().numpy()
    y_pred = y_prob.argmax(axis=1).astype(np.int64)
    return compute_metrics(y_true=y, y_pred=y_pred, y_prob=y_prob, threshold=threshold)


patchtst_full = compute_threshold_metrics(patchtst_model, X_test, y_test, DEVICE, THRESHOLD)
itransformer_full = compute_threshold_metrics(itransformer_model, X_test, y_test, DEVICE, THRESHOLD)
cnn_full = compute_threshold_metrics(cnn_model, X_test, y_test, DEVICE, THRESHOLD)

precision_key = f"precision_at_{THRESHOLD}"
print(f"=== Precision@{THRESHOLD} ===")
print(f"  PatchTST:     {patchtst_full[precision_key]:.4f} (高信頼度サンプル数: {patchtst_full['n_high_conf']})")
print(f"  iTransformer: {itransformer_full[precision_key]:.4f} (高信頼度サンプル数: {itransformer_full['n_high_conf']})")
print(f"  CNN:          {cnn_full[precision_key]:.4f} (高信頼度サンプル数: {cnn_full['n_high_conf']})")

In [ ]:
# 全メトリクスを統合した比較 DataFrame の生成と CSV 保存（AC-015）
from datetime import datetime

full_results = {
    "patchtst": patchtst_full,
    "itransformer": itransformer_full,
    "cnn": cnn_full,
}

df_full = compare_models(full_results)

# CSV 保存（AC-015: テストデータに対して全メトリクスが CSV ファイルに出力される）
WORKING_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = WORKING_DIR / f"evaluation_results_all_{timestamp}.csv"
df_full.to_csv(csv_path, index=False)
print(f"AC-015 OK: 評価結果を保存しました: {csv_path}")

print("\n=== 全メトリクス比較表 ===")
print(df_full.to_string(index=False))

In [ ]:
# PRD 成功基準の検証
print("=" * 60)
print("PRD 成功基準の確認")
print("=" * 60)

# CNN ベースラインの F1 スコア（UP/DOWN 平均）
cnn_f1_up = cnn_full["f1_up"]
cnn_f1_down = cnn_full["f1_down"]
cnn_f1_mean = (cnn_f1_up + cnn_f1_down) / 2

print("\n[CNN ベースライン]")
print(f"  F1_UP:   {cnn_f1_up:.4f}")
print(f"  F1_DOWN: {cnn_f1_down:.4f}")
print(f"  F1 平均: {cnn_f1_mean:.4f}")

# 基準 1: UP/DOWN F1 スコアが CNN より +5pt 以上
F1_IMPROVEMENT_THRESHOLD = 0.05
print(f"\n[基準 1] UP/DOWN F1 が CNN より +{F1_IMPROVEMENT_THRESHOLD*100:.0f}pt 以上")
for model_name, metrics in [("patchtst", patchtst_full), ("itransformer", itransformer_full)]:
    f1_up = metrics["f1_up"]
    f1_down = metrics["f1_down"]
    f1_mean = (f1_up + f1_down) / 2
    improvement = f1_mean - cnn_f1_mean
    passed = improvement >= F1_IMPROVEMENT_THRESHOLD
    status = "PASS" if passed else "FAIL"
    print(f"  {model_name}: F1_UP={f1_up:.4f}, F1_DOWN={f1_down:.4f}, "
          f"F1_mean={f1_mean:.4f}, improvement={improvement:+.4f} [{status}]")

# 基準 2: Precision@0.8 >= 70%
PRECISION_THRESHOLD = 0.70
print(f"\n[基準 2] Precision@{THRESHOLD} >= {PRECISION_THRESHOLD*100:.0f}%")
for model_name, metrics in [
    ("patchtst", patchtst_full),
    ("itransformer", itransformer_full),
    ("cnn", cnn_full),
]:
    prec = metrics[precision_key]
    n_high = metrics["n_high_conf"]
    import math
    if math.isnan(prec):
        print(f"  {model_name}: Precision@{THRESHOLD}=N/A (高信頼度サンプルなし) [SKIP]")
    else:
        passed = prec >= PRECISION_THRESHOLD
        status = "PASS" if passed else "FAIL"
        print(f"  {model_name}: Precision@{THRESHOLD}={prec:.4f} (n={n_high}) [{status}]")

print("\n=" * 60)

In [ ]:
# 最良モデルの選定
# 評価基準: F1_UP と F1_DOWN の平均が最も高いモデルを採用
model_scores = {
    "patchtst": (patchtst_full["f1_up"] + patchtst_full["f1_down"]) / 2,
    "itransformer": (itransformer_full["f1_up"] + itransformer_full["f1_down"]) / 2,
    "cnn": (cnn_full["f1_up"] + cnn_full["f1_down"]) / 2,
}

best_model_name = max(model_scores, key=lambda k: model_scores[k])
best_score = model_scores[best_model_name]

print("=== モデルスコア（F1_UP + F1_DOWN の平均）===")
for name, score in sorted(model_scores.items(), key=lambda x: x[1], reverse=True):
    marker = " ← 最良" if name == best_model_name else ""
    print(f"  {name}: {score:.4f}{marker}")

print(f"\n採用候補モデル: {best_model_name} (F1_mean={best_score:.4f})")
print("\n全 AC 確認完了。Kaggle commit mode 実行成功。")